In [ ]:
!python --version


Python 3.11.13


In [ ]:
!python --version


Python 3.11.13


In [ ]:
!pip install -q pycaret

In [ ]:
!pip install -q m2cgen

In [ ]:
import pandas as pd
import numpy as np


In [ ]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

In [ ]:
df_train.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,37,admin.,single,secondary,no,483,no,no,cellular,31,jul,11,11,-1,0,unknown,no
1,53,management,divorced,tertiary,no,253,no,no,cellular,27,aug,202,6,-1,0,unknown,no
2,55,management,married,unknown,no,1504,yes,no,cellular,31,aug,186,3,101,2,success,yes
3,33,services,married,secondary,no,0,yes,no,cellular,14,jul,163,2,-1,0,unknown,no
4,27,admin.,single,primary,no,528,yes,yes,cellular,18,may,267,2,311,2,failure,no


In [ ]:
df_train['y'].value_counts()

,count
y,
no,31937
yes,4231


In [ ]:
from pycaret.classification import *

In [ ]:
from sklearn.metrics import f1_score

In [ ]:
s = setup(data=df_train,target='y')

,Description,Value
0,Session id,6189
1,Target,y
2,Target type,Binary
3,Target mapping,"no: 0, yes: 1"
4,Original data shape,"(36168, 17)"
5,Transformed data shape,"(36168, 49)"
6,Transformed train set shape,"(25317, 49)"
7,Transformed test set shape,"(10851, 49)"
8,Numeric features,7
9,Categorical features,9


In [ ]:
def macro_f1(y, y_pred, **kwargs):
  return f1_score(y,y_pred,average='macro')

add_metric('macro_f1','Macro F1',macro_f1)
print('setting up custom metric completed')


setting up custom metric completed


In [ ]:
multimodel = ['lr','knn','dt','rf','ada']
best_model = compare_models(include=multimodel,fold=10,sort='Macro F1',verbose=False)

/usr/local/lib/python3.11/dist-packages/pycaret/internal/pycaret_experiment/supervised_experiment.py:339: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(highlight_cols, subset=["TT (Sec)"])


In [ ]:
metrics = pull()
metrics.loc[:,['Macro F1']]

,Macro F1
rf,0.7081
dt,0.7039
ada,0.7036
lr,0.6911
knn,0.6428


In [ ]:
metrics.columns

Index(['Model', 'Accuracy', 'AUC', 'Recall', 'Prec.', 'F1', 'Kappa', 'MCC',
       'Macro F1', 'TT (Sec)'],
      dtype='object')

In [ ]:
type(best_model).__name__

'RandomForestClassifier'

In [ ]:
custom_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': [None, 'balanced']
}
tuned_model = tune_model(best_model,custom_grid=custom_grid,optimize='Macro F1',fold=10,verbose=False)


In [ ]:
tune_metrics = pull()

In [ ]:
tune_metrics.loc[['Mean'],['Macro F1']]

,Macro F1
Fold,
Mean,0.772


In [ ]:
for k,best_v in best_model.get_params().items():
  tuned_v = tuned_model.get_params().get(k)
  if best_v != tuned_v:
    print(f"{k} : {best_v} --> {tuned_v}")


class_weight : None --> balanced
min_samples_leaf : 1 --> 4
min_samples_split : 2 --> 10


In [ ]:
final_model = finalize_model(tuned_model)

In [ ]:
prediction = predict_model(final_model,data=df_test)

In [ ]:
submission = pd.DataFrame({
    'id':df_test['id'],
    'y':prediction['prediction_label']
})

In [ ]:
submission.to_csv('predict.txt',index=False)

In [ ]:
import joblib

In [ ]:
model_bundle = {
    'model': final_model,
    'features': df_train.columns.tolist(),
    'target': 'y',
    'classes': final_model.classes_.tolist()
}

In [ ]:
joblib.dump(model_bundle, 'bank_dt_joblib.joblib')

['bank_dt_joblib.joblib']